# Integrate spatial RNA (rep1) with Zhuang-ABCA-1 MERFISH reference
Transfer cell type annotations (class, subclass, subclass_label, supertype) via Seurat CCA + kNN

- Reference: Zhuang-ABCA-1.080.h5ad (37,068 cells, 1,122 MERFISH genes)
- Query: brain rep1 spatial RNA (6,275 spots), using full gene expression from raw 10x matrix

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import os
import anndata
import scanpy as sc
import pynndescent

from scipy.sparse import issparse
from sklearn.preprocessing import normalize, OneHotEncoder
from sklearn.decomposition import PCA

from ALLCools.plot import *
from ALLCools.clustering import *
from ALLCools.integration.seurat_class import SeuratIntegration

import seaborn as sns

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

In [ ]:
# Paths
REF_PATH = '/large_storage/zhoulab/jche/project/spatialHiC/05.integrate_ref_brain/integrate_based_ZJT/09.merfish_subset_integrate/02.split_ref/split_output/Zhuang-ABCA-1.080_subset_neuron.h5ad'
QUERY_ANNOT_PATH = '/large_storage/zhoulab/jche/project/spatialHiC/05.integrate_ref_brain/integrate_based_ZJT/09.merfish_subset_integrate/03.split_query/split_output/rep1_neuron.h5ad'
# QUERY_RAW_PATH = '/large_storage/zhoulab/jche/project/spatialHiC/data/rna/GSM9320917_mouse_brain_r1_RNA/raw'
OUTDIR = '/large_storage/zhoulab/jche/project/spatialHiC/05.integrate_ref_brain/integrate_based_ZJT/09.merfish_subset_integrate/04.round2/neuron-no_anchor_filter/integrate_output/'
os.makedirs(OUTDIR, exist_ok=True)

# Parameters
NPC = 50        # number of PCs
NCC = 50        # number of CCA components
K_NN = 25       # k for kNN annotation transfer
SD = 1          # distance weighting parameter

In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f"{name}_{i}"] = adata.obsm[f"X_{name}"][:, i]
    return adata

## 1. Load reference

In [ ]:
adata_ref = sc.read_h5ad(REF_PATH)
print(f"Reference: {adata_ref.shape}")
print(f"Annotation levels:")
for col in ['class', 'subclass', 'subclass_label', 'supertype']:
    print(f"  {col}: {adata_ref.obs[col].nunique()} unique")
adata_ref.obs[['class', 'subclass', 'subclass_label', 'supertype']].head(10)

In [ ]:
# Compute total counts per cell (MERFISH raw counts)
adata_ref.obs['total_counts'] = np.array(adata_ref.X.sum(axis=1)).flatten()
print(f"Total counts per cell - median: {np.median(adata_ref.obs['total_counts']):.0f}, "
      f"min: {adata_ref.obs['total_counts'].min():.0f}, max: {adata_ref.obs['total_counts'].max():.0f}")

## 2. Load query (full gene expression from raw 10x, subset to annotated cells)

In [ ]:
# Load the annotated query to get cell barcodes and metadata
# import h5py
# from anndata._io.specs import read_elem

# f = h5py.File(QUERY_ANNOT_PATH, 'r')
# obs_annot = read_elem(f['obs'])
# f.close()
# query_barcodes = obs_annot.index.tolist()
# print(f"Annotated query cells: {len(query_barcodes)}")
# print(f"First 3 barcodes: {query_barcodes[:3]}")

In [ ]:
# Load full gene expression from raw 10x matrix
# adata_raw = sc.read_10x_mtx(QUERY_RAW_PATH, var_names='gene_ids', make_unique=True)
# print(f"Raw 10x matrix: {adata_raw.shape}")
# print(f"Raw barcodes[:3]: {adata_raw.obs.index[:3].tolist()}")
# print(f"Raw var[:3]: {adata_raw.var.index[:3].tolist()}")
# print(f"Raw var columns: {adata_raw.var.columns.tolist()}")

In [ ]:
# Strip -1 suffix from 10x barcodes if present
# adata_raw.obs.index = adata_raw.obs.index.str.replace('-1$', '', regex=True)

# # Subset to annotated query cells
# common_barcodes = [b for b in query_barcodes if b in adata_raw.obs.index]
# print(f"Barcodes found in raw matrix: {len(common_barcodes)} / {len(query_barcodes)}")

# adata_qry = adata_raw[common_barcodes].copy()

# # Transfer metadata from annotated query
# adata_qry.obs = obs_annot.loc[common_barcodes].copy()

adata_qry = sc.read_h5ad(QUERY_ANNOT_PATH)
print(f"Query: {adata_qry.shape}")

In [ ]:
# Compute total counts for query from raw matrix
adata_qry.obs['n_counts_raw'] = np.array(adata_qry.X.sum(axis=1)).flatten()
print(f"Query total counts - median: {np.median(adata_qry.obs['n_counts_raw']):.0f}, "
      f"min: {adata_qry.obs['n_counts_raw'].min():.0f}, max: {adata_qry.obs['n_counts_raw'].max():.0f}")

## 3. Match genes by Ensembl ID

In [ ]:
# Ref var index: Ensembl IDs (no version), e.g. ENSMUSG00000024798
# Query var index: gene symbols; Ensembl IDs are in var['gene_ids']
# Match on Ensembl IDs: strip version suffix from query's gene_ids column

adata_qry.var['original_id'] = adata_qry.var.index.copy()
adata_qry.var['ensembl_id'] = adata_qry.var['gene_ids'].str.split('.').str[0]

# Handle potential duplicates after stripping version
dup_mask = adata_qry.var['ensembl_id'].duplicated(keep='first')
if dup_mask.sum() > 0:
    print(f"Removing {dup_mask.sum()} duplicate gene IDs after version stripping")
    adata_qry = adata_qry[:, ~dup_mask].copy()

# Set query var index to Ensembl IDs for intersection
adata_qry.var.index = adata_qry.var['ensembl_id']

common_genes = adata_ref.var.index.intersection(adata_qry.var.index)
print(f"Reference genes: {adata_ref.shape[1]}")
print(f"Query genes (raw): {adata_qry.shape[1]}")
print(f"Common genes: {len(common_genes)}")

In [ ]:
adata_ref = adata_ref[:, common_genes].copy()
adata_qry = adata_qry[:, common_genes].copy()
print(f"After subsetting - Ref: {adata_ref.shape}, Query: {adata_qry.shape}")

## 4. Normalize reference (median-ratio + log1p)

In [ ]:
adata_ref.X = adata_ref.X.tocsr().astype(np.float64)
median_counts_ref = np.median(adata_ref.obs['total_counts'])
adata_ref.X.data = adata_ref.X.data / np.repeat(
    adata_ref.obs['total_counts'].values, np.diff(adata_ref.X.indptr)
) * median_counts_ref
sc.pp.log1p(adata_ref)
print(f"Reference normalized (median counts = {median_counts_ref:.0f})")

## 5. Feature selection: cluster-enriched features

In [ ]:
# cluster_col = 'subclass_label'
# nclust = len(adata_ref.obs[cluster_col].unique())
# ngene = max(100, 2000 // nclust)
# print(f"{nclust} clusters, selecting top {ngene} genes per cluster")
# print(f"(Total available genes: {adata_ref.shape[1]})")

# cluster_enriched_features(
#     adata_ref, cluster_col=cluster_col,
#     top_n=ngene, alpha=0.05, stat_plot=False, method='rna'
# )
# selected_genes = adata_ref.var.index[adata_ref.var[f'{cluster_col}_enriched_features']].tolist()
# print(f"Selected {len(selected_genes)} cluster-enriched features out of {adata_ref.shape[1]}")
# np.savetxt(f'{OUTDIR}rep1_Zhuang_ABCA1-cef.txt', selected_genes, fmt='%s')

In [ ]:
# adata_ref = adata_ref[:, selected_genes].copy()

## 6. Normalize query, subset to selected features

In [ ]:
adata_qry.X = adata_qry.X.tocsr().astype(np.float64)
adata_qry.X.data = adata_qry.X.data / np.repeat(
    adata_qry.obs['n_counts_raw'].values, np.diff(adata_qry.X.indptr)
) * median_counts_ref
sc.pp.log1p(adata_qry)
# adata_qry = adata_qry[:, selected_genes].copy()
# print(f"Features for integration: {len(selected_genes)}")

## 7. PCA on reference, project query

In [ ]:
X_ref = adata_ref.X.toarray() if issparse(adata_ref.X) else np.array(adata_ref.X)
X_qry = adata_qry.X.toarray() if issparse(adata_qry.X) else np.array(adata_qry.X)

pca_model = PCA(n_components=min(100, X_ref.shape[1] - 1, X_ref.shape[0] - 1),
                svd_solver='arpack', random_state=0)
adata_ref.obsm['pca_all'] = pca_model.fit_transform(X_ref)
adata_ref.obsm['X_pca'] = normalize(adata_ref.obsm['pca_all'][:, :NPC], axis=1)
adata_ref.obs['dataset'] = 'reference'

adata_qry.obsm['pca_all'] = pca_model.transform(X_qry)
adata_qry.obsm['X_pca'] = normalize(adata_qry.obsm['pca_all'][:, :NPC], axis=1)
adata_qry.obs['dataset'] = 'query'
print(f"PCA done: {NPC} components")

## 8. Seurat CCA integration

In [ ]:
adata_list = [adata_ref, adata_qry]

integrator = SeuratIntegration()
integrator.find_anchor(
    adata_list,
    k_local=None,
    key_local='X_pca',
    k_anchor=5,
    key_anchor='X',
    dim_red='pca-cca',
    max_cc_cells=50000,
    k_score=30,
    k_filter=None,
    scale_list=[True, True],
    n_components=NCC,
    n_features=200,
    alignments=[[[0], [1]]]
)
anchor = integrator.anchor[(0, 1)]
print(f"Found {anchor.shape[0]} anchors")

In [ ]:
corrected = integrator.integrate(
    key_correct='X_pca',
    row_normalize=True,
    n_components=NPC,
    k_weight=100,
    sd=SD,
    alignments=[[[0], [1]]]
)
print("Integration done")

## 9. Build merged embedding

In [ ]:
adata_merge = anndata.AnnData(
    X=np.ones((sum(x.shape[0] for x in adata_list), 1)),
    obs=pd.concat([x.obs for x in adata_list], axis=0)
)
adata_merge.obsm['X_pca_corrected'] = normalize(
    np.concatenate(corrected, axis=0), axis=1
)
adata_merge.obsm['X_pca'] = adata_merge.obsm['X_pca_corrected'].copy()
print(f"Merged: {adata_merge.shape}")

## 10. Leiden clustering + tSNE

In [ ]:
sc.pp.neighbors(adata_merge, use_rep='X_pca', n_neighbors=25,
                random_state=0, metric='cosine')
sc.tl.leiden(adata_merge, resolution=2.0, random_state=0, flavor='igraph')
print(f"Leiden clusters: {adata_merge.obs['leiden'].nunique()}")

In [ ]:
tsne(adata_merge, obsm='X_pca_corrected', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
dump_embedding(adata_merge, 'tsne')

## 11. kNN annotation transfer

In [ ]:
ref_mask = adata_merge.obs['dataset'] == 'reference'
qry_mask = adata_merge.obs['dataset'] == 'query'

n_ref = ref_mask.sum()
n_qry = qry_mask.sum()
k = min(K_NN, n_ref)

# Build NN index on reference cells
index = pynndescent.NNDescent(
    adata_merge.obsm['X_pca'][ref_mask],
    metric='euclidean', n_neighbors=min(50, n_ref),
    random_state=0, n_jobs=-1
)
G, D = index.query(adata_merge.obsm['X_pca'][qry_mask], k=k)

# Distance weighting (exponential kernel)
cellfilter = (D[:, -1] == 0)
D = (1 - D / D[:, -1][:, None])
D[cellfilter] = 1
D = 1 - np.exp(-D * (SD**2) / 4)
D = D / (np.sum(D, axis=1) + 1e-6)[:, None]

In [ ]:
# Transfer each annotation level
for label_col in ['class', 'subclass', 'subclass_label', 'supertype']:
    print(f"Transferring '{label_col}'...")
    enc = OneHotEncoder()
    ref_labels = adata_merge.obs.loc[ref_mask, [label_col]].values.astype(str)
    labelref = enc.fit_transform(ref_labels)

    result_df = pd.DataFrame(
        index=adata_merge.obs.index[qry_mask],
        columns=[f'{label_col}_pred', f'{label_col}_score']
    )

    chunk_size = 50000
    for chunk_start in range(0, n_qry, chunk_size):
        chunk_end = min(chunk_start + chunk_size, n_qry)
        D_chunk = D[chunk_start:chunk_end]
        G_chunk = G[chunk_start:chunk_end]

        neighbor_labels = labelref[G_chunk.flatten()].toarray().reshape(
            (-1, k, len(enc.categories_[0]))
        )
        prob = (D_chunk[:, :, None] * neighbor_labels).sum(axis=1)
        prob_df = pd.DataFrame(
            prob, columns=enc.categories_[0],
            index=adata_merge.obs.index[qry_mask][chunk_start:chunk_end]
        )
        result_df.loc[prob_df.index, f'{label_col}_pred'] = prob_df.idxmax(axis=1).values
        result_df.loc[prob_df.index, f'{label_col}_score'] = prob_df.max(axis=1).values

    adata_merge.obs.loc[qry_mask, f'{label_col}_pred'] = result_df[f'{label_col}_pred'].values
    adata_merge.obs.loc[qry_mask, f'{label_col}_score'] = result_df[f'{label_col}_score'].astype(float).values
    print(f"  Unique {label_col} predictions: {result_df[f'{label_col}_pred'].nunique()}")

## 12. Save merged result

In [ ]:
# Save merged h5ad - convert a copy to avoid mutating adata_merge before visualization
adata_save = adata_merge.copy()
for col in adata_save.obs.columns:
    if adata_save.obs[col].dtype == 'object' or adata_save.obs[col].apply(type).nunique() > 1:
        adata_save.obs[col] = adata_save.obs[col].astype(str)
adata_save.write_h5ad(f'{OUTDIR}rep1_Zhuang_ABCA1-neuron_adata_merged.h5ad')
del adata_save
print(f"Saved: {OUTDIR}rep1_Zhuang_ABCA1-neuron_adata_merged.h5ad")

## 13. Visualization

In [ ]:
# Build color palette from reference
ref_obs = adata_merge.obs.loc[ref_mask]
subclass_palette = dict(zip(ref_obs['subclass_label'], ref_obs['subclass_color']))
class_palette = dict(zip(ref_obs['class'], ref_obs['class_color']))
supertype_palette = dict(zip(ref_obs['supertype'], ref_obs['supertype_color']))

In [ ]:
ds = 0.5
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=300, constrained_layout=True)

# Top-left: reference colored by class
tmp = adata_merge.obs.loc[ref_mask].copy()
ax = axes[0, 0]
ax.scatter(adata_merge.obs[f'{coord_base}_0'], adata_merge.obs[f'{coord_base}_1'],
           c='#e0e0e0', edgecolors='none', s=ds, alpha=1, rasterized=True)
_ = categorical_scatter(data=tmp, ax=ax, coord_base=coord_base, s=ds,
                        hue='class', labelsize=6, max_points=None,
                        palette=class_palette,
                        scatter_kws={'rasterized': True},
                        show_legend=True, legend_kws={'ncol': 1, 'fontsize': 5})
ax.set_title('Reference: class')

# Top-right: leiden
tmp = adata_merge.obs.copy()
ax = axes[0, 1]
_ = categorical_scatter(data=tmp, ax=ax, coord_base=coord_base, s=ds,
                        hue='leiden', text_anno='leiden', labelsize=8,
                        max_points=None, palette='tab20',
                        scatter_kws={'rasterized': True})
ax.set_title('Leiden clusters')

# Bottom-left: reference subclass_label
tmp = adata_merge.obs.loc[ref_mask].copy()
ax = axes[1, 0]
ax.scatter(adata_merge.obs[f'{coord_base}_0'], adata_merge.obs[f'{coord_base}_1'],
           c='#e0e0e0', edgecolors='none', s=ds, alpha=1, rasterized=True)
count = tmp['subclass_label'].value_counts()
tmp_filt = tmp.loc[tmp['subclass_label'].isin(count.index[count >= 50])]
_ = categorical_scatter(data=tmp_filt, ax=ax, coord_base=coord_base, s=ds,
                        hue='subclass_label', palette=subclass_palette,
                        labelsize=5, max_points=None,
                        scatter_kws={'rasterized': True})
ax.set_title('Reference: subclass_label')

# Bottom-right: query predicted subclass_label
tmp = adata_merge.obs.loc[qry_mask].copy()
tmp = tmp.dropna(subset=['subclass_label_pred'])
tmp = tmp[tmp['subclass_label_pred'] != 'nan']
ax = axes[1, 1]
ax.scatter(adata_merge.obs[f'{coord_base}_0'], adata_merge.obs[f'{coord_base}_1'],
           c='#e0e0e0', edgecolors='none', s=ds, alpha=0.5, rasterized=True)
_ = categorical_scatter(data=tmp, ax=ax, coord_base=coord_base, s=ds*5,
                        hue='subclass_label_pred', palette=subclass_palette,
                        labelsize=5, max_points=None,
                        scatter_kws={'rasterized': True})
ax.set_title('Query: predicted subclass_label')

plt.savefig(f'{OUTDIR}rep1_Zhuang_ABCA1-neuron_tsne_overview.pdf', transparent=True)
plt.show()

In [ ]:
ds = 0.5
coord_base = 'tsne'

fig, ax = plt.subplots(figsize=(12, 8), dpi=300, constrained_layout=True)

tmp = adata_merge.obs.copy()

ax.scatter(adata_merge.obs[f'{coord_base}_0'], adata_merge.obs[f'{coord_base}_1'],
           c='#e0e0e0', edgecolors='none', s=ds, alpha=1, rasterized=True)
count = tmp['subclass_label'].value_counts()
tmp = tmp.loc[(tmp['subclass_label'].isin(count.index[count >= 100])) &
              (tmp['subclass_label'].isin(subclass_palette.keys()))]

_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='subclass_label',
                        s=ds,
                        palette=subclass_palette,
                        labelsize=8,
                        max_points=None,
                        scatter_kws={'rasterized': True},
                       )

# Add text labels manually without bbox
for label, sub_df in tmp.groupby('subclass_label'):
    if str(label).lower() in ['', 'nan']:
        continue
    _x = sub_df[f'{coord_base}_0'].median()
    _y = sub_df[f'{coord_base}_1'].median()
    ax.text(_x, _y, str(label), fontsize=8, fontweight='bold',
            ha='center', va='center')

ax.set_title('Merged tSNE - subclass_label (count >= 100)')

In [ ]:
# Classify ALL cells (ref + query) into neuron / non-neuron on merged adata
def classify_cell(row):
    if row['dataset'] == 'reference':
        sc_label = row['subclass_label']
    else:
        sc_label = row['subclass_label_pred']
    if pd.isna(sc_label) or sc_label == 'nan':
        return 'unknown'
    return 'non-neuron' if sc_label.endswith('NN') else 'neuron'

adata_merge.obs['cell_type'] = adata_merge.obs.apply(classify_cell, axis=1)
print(adata_merge.obs.groupby('dataset')['cell_type'].value_counts())

# Plot t-SNE from merged adata colored by neuron/non-neuron, split by dataset
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

tsne = adata_merge.obsm['X_tsne']
colors = {'neuron': '#1f77b4', 'non-neuron': '#ff7f0e', 'unknown': '#cccccc'}

# Panel 1: All cells
for ct in ['unknown', 'non-neuron', 'neuron']:
    mask = adata_merge.obs['cell_type'] == ct
    if mask.sum() == 0:
        continue
    axes[0].scatter(tsne[mask, 0], tsne[mask, 1], c=colors[ct], s=1, alpha=0.3, label=ct, rasterized=True)
axes[0].set_title('rep1 - All cells')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
axes[0].legend(markerscale=5)

# Panel 2: Reference only
ref_m = adata_merge.obs['dataset'] == 'reference'
for ct in ['non-neuron', 'neuron']:
    mask = ref_m & (adata_merge.obs['cell_type'] == ct)
    if mask.sum() == 0:
        continue
    axes[1].scatter(tsne[mask, 0], tsne[mask, 1], c=colors[ct], s=1, alpha=0.3, label=ct, rasterized=True)
axes[1].set_title('rep1 - Reference (Zhuang-ABCA-1)')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend(markerscale=5)

# Panel 3: Query only
qm = adata_merge.obs['dataset'] == 'query'
for ct in ['unknown', 'non-neuron', 'neuron']:
    mask = qm & (adata_merge.obs['cell_type'] == ct)
    if mask.sum() == 0:
        continue
    axes[2].scatter(tsne[mask, 0], tsne[mask, 1], c=colors[ct], s=1, alpha=0.3, label=ct, rasterized=True)
axes[2].set_title('rep1 - Query')
axes[2].set_xlabel('t-SNE 1')
axes[2].set_ylabel('t-SNE 2')
axes[2].legend(markerscale=5)

plt.tight_layout()
# plt.savefig(f'{out_dir}/rep1_tsne_neuron_vs_nonneuron.pdf', dpi=150, bbox_inches='tight')
# plt.savefig(f'{out_dir}/rep1_tsne_neuron_vs_nonneuron.png', dpi=150, bbox_inches='tight')
plt.show()
# print(f"Saved: {out_dir}/rep1_tsne_neuron_vs_nonneuron.pdf / .png")

In [ ]:
# Spatial plots of predicted annotations
tmp = adata_merge.obs.loc[qry_mask].copy()
tmp = tmp.dropna(subset=['subclass_label_pred'])
tmp = tmp[tmp['subclass_label_pred'] != 'nan']
tmp[['spatial_0', 'spatial_1']] = tmp[['array_row', 'array_col']]
coord_base = 'spatial'
ds = 5

fig, axes = plt.subplots(2, 2, figsize=(12, 10), dpi=300, constrained_layout=True)

# subclass_label_pred
ax = axes[0, 0]
ax.axis('equal')
count = tmp['subclass_label_pred'].value_counts()
leg = np.sort(count.index[count >= 10])
tmp_filt = tmp.loc[tmp['subclass_label_pred'].isin(leg)]
_ = categorical_scatter(data=tmp_filt, ax=ax, coord_base=coord_base, s=ds,
                        hue='subclass_label_pred', palette=subclass_palette,
                        labelsize=5, max_points=None,
                        scatter_kws={'rasterized': True},
                        show_legend=True, legend_kws={'ncol': 2, 'fontsize': 4})
ax.set_title('Predicted subclass_label')

# class_pred
ax = axes[0, 1]
ax.axis('equal')
tmp_class = tmp.dropna(subset=['class_pred'])
tmp_class = tmp_class[tmp_class['class_pred'] != 'nan']
_ = categorical_scatter(data=tmp_class, ax=ax, coord_base=coord_base, s=ds,
                        hue='class_pred', palette=class_palette,
                        labelsize=5, max_points=None,
                        scatter_kws={'rasterized': True},
                        show_legend=True, legend_kws={'ncol': 1, 'fontsize': 5})
ax.set_title('Predicted class')

# supertype_pred
ax = axes[1, 0]
ax.axis('equal')
tmp_st = tmp.dropna(subset=['supertype_pred'])
tmp_st = tmp_st[tmp_st['supertype_pred'] != 'nan']
count_st = tmp_st['supertype_pred'].value_counts()
leg_st = np.sort(count_st.index[count_st >= 10])
tmp_filt_st = tmp_st.loc[tmp_st['supertype_pred'].isin(leg_st)]
_ = categorical_scatter(data=tmp_filt_st, ax=ax, coord_base=coord_base, s=ds,
                        hue='supertype_pred', palette=supertype_palette,
                        labelsize=4, max_points=None,
                        scatter_kws={'rasterized': True},
                        show_legend=True, legend_kws={'ncol': 2, 'fontsize': 3})
ax.set_title('Predicted supertype')

# n_counts
ax = axes[1, 1]
ax.axis('equal')
_ = continuous_scatter(ax=ax, data=tmp,
                       hue=np.log10(tmp['n_counts_raw'] + 1),
                       coord_base=coord_base, max_points=None,
                       labelsize=8, s=ds,
                       scatter_kws={'rasterized': True})
ax.set_title('log10(n_counts + 1)')

plt.savefig(f'{OUTDIR}rep1_Zhuang_ABCA1-neuron_spatial_predictions.pdf', transparent=True)
plt.show()

In [ ]:
# Spatial subclass_label by groups of 20
nplot = (len(leg) - 1) // 20 + 1
ncol = 2
nrow = (nplot - 1) // ncol + 1
coord_base = 'spatial'
ds = 5

fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 5, nrow * 3), dpi=300, constrained_layout=True)
if nrow * ncol == 1:
    axes = np.array([axes])
for i in range(nplot):
    legtmp = leg[(i * 20): ((i + 1) * 20)]
    tmp0 = tmp.loc[tmp['subclass_label_pred'].isin(legtmp)]
    ax = axes.flatten()[i]
    ax.axis('equal')
    ax.scatter(tmp[f'{coord_base}_0'], tmp[f'{coord_base}_1'],
              c='#e0e0e0', edgecolors='none', s=ds, alpha=0.5, rasterized=True)
    _ = categorical_scatter(data=tmp0, ax=ax, coord_base=coord_base, s=ds,
                            hue='subclass_label_pred', palette='tab20',
                            labelsize=6, max_points=None,
                            scatter_kws={'rasterized': True},
                            show_legend=True, legend_kws={'ncol': 1, 'fontsize': 6})

for ax in axes.flatten()[nplot:]:
    ax.axis('off')

plt.savefig(f'{OUTDIR}rep1_Zhuang_ABCA1-neuron_spatial_subclass_detail.pdf', transparent=True)
plt.show()

## 14. Save query annotations

In [ ]:
# Convert object columns to str for h5ad compatibility, then save
adata_merge.obs.loc[:, adata_merge.obs.dtypes == 'object'] = adata_merge.obs.loc[:, adata_merge.obs.dtypes == 'object'].astype(str)

query_obs = adata_merge.obs.loc[qry_mask].copy()
query_obs.to_csv(f'{OUTDIR}rep1_Zhuang_ABCA1-neuron_query_annotations.csv')
print(f"Saved: {OUTDIR}rep1_Zhuang_ABCA1-neuron_query_annotations.csv")
print(f"\nPrediction summary:")
for col in ['class_pred', 'subclass_pred', 'subclass_label_pred', 'supertype_pred']:
    print(f"  {col}: {query_obs[col].nunique()} unique values")
query_obs[['class_pred', 'class_score', 'subclass_label_pred', 'subclass_label_score', 'supertype_pred', 'supertype_score']].head(10)